<a href="https://colab.research.google.com/github/AdviktheGreat/gdmprojects/blob/main/numTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a Decoder-Only Transformer for Addition with JAX/Flax
This notebook demonstrates how to build and train a small transformer from scratch to solve simple addition problems.

In [ ]:
import jax
import jax.numpy as jnp
import flax
from flax import linen as nn
from flax.training import train_state
import optax
import numpy as np
from typing import Any, Tuple

# Device check
print(f"JAX default backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

## 2. Vocabulary and Tokenization

In [ ]:
CHARS = "0123456789 +=#"
# # is used as a padding token
char_to_id = {c: i for i, c in enumerate(CHARS)}
id_to_char = {i: c for c, i in char_to_id.items()}
VOCAB_SIZE = len(CHARS)
PAD_ID = char_to_id['#']

def encode(s: str) -> np.ndarray:
    return np.array([char_to_id[c] for c in s], dtype=np.int32)

def decode(ids) -> str:
    return "".join([id_to_char[int(i)] for i in ids])

## 3. Synthetic Dataset Generation

In [ ]:
MAX_DIGITS = 3
# Max expression: '999+999=1998' (12 chars). Let's use 16 for safety.
MAX_LEN = 16

def generate_dataset(num_samples=10000, seed=42):
    rng = np.random.default_rng(seed)
    data = []
    for _ in range(num_samples):
        a, b = rng.integers(0, 10**MAX_DIGITS, size=2)
        expr = f"{a}+{b}={a+b}"
        encoded = encode(expr)
        # Pad with PAD_ID
        padded = np.full((MAX_LEN,), PAD_ID, dtype=np.int32)
        padded[:len(encoded)] = encoded
        data.append(padded)
    return np.array(data)

dataset = generate_dataset(20000)
split = int(0.9 * len(dataset))
train_ds, val_ds = dataset[:split], dataset[split:]

## 4. Model Definition

In [ ]:
class SelfAttention(nn.Module):
    num_heads: int
    head_dim: int

    @nn.compact
    def __call__(self, x, mask=None):
        b, l, d = x.shape
        q = nn.Dense(self.num_heads * self.head_dim)(x)
        k = nn.Dense(self.num_heads * self.head_dim)(x)
        v = nn.Dense(self.num_heads * self.head_dim)(x)

        q = q.reshape(b, l, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(b, l, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(b, l, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        scores = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)
        if mask is not None:
            scores = jnp.where(mask, scores, -1e9)

        attn = jax.nn.softmax(scores, axis=-1)
        out = jnp.matmul(attn, v).transpose(0, 2, 1, 3).reshape(b, l, -1)
        return nn.Dense(d)(out)

class TransformerBlock(nn.Module):
    num_heads: int
    head_dim: int
    mlp_dim: int

    @nn.compact
    def __call__(self, x, mask=None):
        # Pre-norm
        h = nn.LayerNorm()(x)
        h = SelfAttention(self.num_heads, self.head_dim)(h, mask)
        x = x + h

        h = nn.LayerNorm()(x)
        h = nn.Dense(self.mlp_dim)(h)
        h = nn.relu(h)
        h = nn.Dense(x.shape[-1])(h)
        return x + h

class DecoderTransformer(nn.Module):
    vocab_size: int
    d_model: int = 256
    num_layers: int = 4
    num_heads: int = 8
    max_len: int = 16

    @nn.compact
    def __call__(self, x):
        b, l = x.shape
        # Token + Positional Embedding
        token_emb = nn.Embed(self.vocab_size, self.d_model)(x)
        pos_emb = self.param('pos_emb', nn.initializers.normal(0.02), (1, self.max_len, self.d_model))
        x = token_emb + pos_emb[:, :l, :]

        # Causal mask
        mask = jnp.tril(jnp.ones((l, l)))
        mask = mask.reshape(1, 1, l, l)

        for _ in range(self.num_layers):
            x = TransformerBlock(self.num_heads, self.d_model // self.num_heads, self.d_model * 4)(x, mask)

        x = nn.LayerNorm()(x)
        return nn.Dense(self.vocab_size)(x)

## 5. Loss, Metrics, and Training Steps

In [ ]:
def cross_entropy_loss(logits, targets):
    one_hot = jax.nn.one_hot(targets, VOCAB_SIZE)
    # Mask out padding from loss calculation
    mask = (targets != PAD_ID).astype(jnp.float32)
    loss = optax.softmax_cross_entropy(logits, one_hot)
    return jnp.sum(loss * mask) / jnp.sum(mask)

@jax.jit
def train_step(state, batch):
    def loss_fn(params):
        # Causal modeling: shift inputs and targets
        logits = state.apply_fn({'params': params}, batch[:, :-1])
        loss = cross_entropy_loss(logits, batch[:, 1:])
        return loss

    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

## 6. Training Loop

In [ ]:
model = DecoderTransformer(vocab_size=VOCAB_SIZE)
params = model.init(jax.random.PRNGKey(0), jnp.zeros((1, MAX_LEN), dtype=jnp.int32))['params']
state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=optax.adam(1e-3)
)

batch_size = 128
num_epochs = 20

for epoch in range(num_epochs):
    np.random.shuffle(train_ds)
    losses = []
    for i in range(0, len(train_ds), batch_size):
        batch = jnp.array(train_ds[i:i+batch_size])
        state, loss = train_step(state, batch)
        losses.append(loss)

    if epoch % 2 == 0:
        print(f"Epoch {epoch}, Loss: {np.mean(losses):.4f}")

## 7. Generation and Evaluation

In [ ]:
def generate(prompt, state, max_tokens=6):
    tokens = encode(prompt).tolist()
    for _ in range(max_tokens):
        input_batch = jnp.array([tokens])
        logits = state.apply_fn({'params': state.params}, input_batch)
        next_token = jnp.argmax(logits[0, -1, :])
        tokens.append(int(next_token))
        if next_token == PAD_ID or id_to_char[int(next_token)] == ' ':
            break
    return decode(tokens)

# Test samples
print(f"Test 12+3: {generate('12+3=', state)}")
print(f"Test 999+1: {generate('999+1=', state)}")